# `lemmas_processor` — Usage Notebook

Demonstrates the core data model and processing logic behind `scripts/lemmas_processor.py`.

**What the script does**
1. **Pass 1 — standard annotation**: for each unannotated corpus token, look up its word form in the dictionary and insert the matching lemma ID.
2. **Pass 2 — unknown words**: tokens still unannotated receive a brand-new ID (prefix `N`).
3. **Normalisation**: rewrites dictionary entries to ensure canonical field order.

**Report files produced**
- `report1_new_id_lines.txt` — lines modified by a newly generated ID
- `report1_5_existing_id_lines.txt` — lines modified by inserting an existing dictionary ID
- `report2_normalized_entries.txt` — dictionary entries reordered to canonical form
- `report3_new_entries.txt` — new dictionary entries created

All cells use the real corpus data in `data/` — no data files are modified.

---
## 0. Setup

In [ ]:
import sys
import os
from collections import defaultdict

sys.path.insert(0, os.path.join('..', 'src'))

DICT_FILE   = os.path.join('..', 'data', 'xml', 'dict', 'dictionary.xml')
CORPUS_FILE = os.path.join('..', 'data', 'xml', 'text', 'EN_01.xml')
TEXT_DIR    = os.path.join('..', 'data', 'xml', 'text')

print("Python:", sys.version.split()[0])


---
## 1. Dictionary Loading

The script parses the dictionary into two flat maps:
- `form_to_lemma` — word form → single lemma ID (used when there is exactly one match)
- `form_to_candidates` — word form → list of candidate dicts `{id, gloss, pos, preceding_tag}` (used by `disambiguate()` when multiple entries share a form)

Below we reproduce the same logic using `oncoj.dictionary`.

In [ ]:
from oncoj.core.dictionary import Dictionary

d = Dictionary.from_file(DICT_FILE)
print(f"Total entries: {len(d)}")

# Build a form → [entry] map
form_to_entries = defaultdict(list)
for entry in d.values():
    for form in entry.get_all('.FORM'):
        form_to_entries[form].append(entry)

# Separate unambiguous vs ambiguous forms
unambiguous = {f: es[0] for f, es in form_to_entries.items() if len(es) == 1}
ambiguous   = {f: es    for f, es in form_to_entries.items() if len(es) > 1}

print(f"Unique forms: {len(form_to_entries)}")
print(f"  Unambiguous (1 entry per form): {len(unambiguous)}")
print(f"  Ambiguous   (2+ entries):       {len(ambiguous)}")

In [ ]:
# Sample of ambiguous forms
print("Sample ambiguous forms (form → candidate IDs):")
for form, entries in list(ambiguous.items())[:8]:
    ids = [e.eid for e in entries]
    glosses = [e.get_first('.GLOSS') or '—' for e in entries]
    print(f"  {form:12s} → {list(zip(ids, glosses))}")

---
## 2. Disambiguation

When multiple dictionary entries share a `.FORM`, the script scores each candidate
against the **preceding syntactic tag** in the corpus line using a heuristic table:

| POS contains | Expected preceding tag | Score |
|---|---|---|
| `noun` | `N`, `N-DVB`, `N-COMP`, … | 3 |
| `verb` | `VB`, `VB-STM`, `VB-INF`, … | 3 |
| `auxiliary` | VAX-*, COP-*, ACP-*, … | 3 |
| `particle` | P-CASE-*, P-TOP, … | 3 |

The candidate with the highest score wins; ties fall back to the first candidate.

In [ ]:
from oncoj.core.tags import strip_disambig

NOUN_TAGS = frozenset({
    'N', 'N-DVB', 'N-COMP', 'N-PRD', 'DVN',
    'PRO-N', 'PRO-ADV', 'NUM', 'NUMCL', 'CL',
})
VERB_TAGS = frozenset({
    'VB', 'VB-STM', 'VB-INF', 'VB-ADN', 'VB-CLS', 'VB-IFC', 'VB-GER',
    'VB-IMP', 'VB-CND', 'VB-CSS', 'VB-NML', 'VB-PRV',
})
AUX_TAGS  = frozenset({
    'ACP', 'ACP-INF', 'ACP-ADN', 'ACP-CLS',
    'COP', 'COP-INF', 'COP-ADN', 'COP-CLS',
})
PART_TAGS = frozenset({
    'P-CASE-GEN', 'P-CASE-DAT', 'P-CASE-ACC', 'P-CASE-COM',
    'P-TOP', 'P-FOC', 'P-CONN', 'P-COMP',
})

def score_candidate(entry, prev_tag: str) -> int:
    pos = (entry.get_first('.POS') or '').lower()
    p = strip_disambig(prev_tag)
    if 'noun' in pos and p in NOUN_TAGS:   return 3
    if 'verb' in pos and p in VERB_TAGS:   return 3
    if 'auxiliary' in pos and p in AUX_TAGS: return 3
    if 'particle' in pos and p in PART_TAGS: return 3
    return 0

def disambiguate(candidates, prev_tag: str):
    scored = [(score_candidate(e, prev_tag), e) for e in candidates]
    scored.sort(key=lambda x: -x[0])
    return scored[0][1]

# Demonstrate: 'ki' is ambiguous (verb vs noun vs auxiliary)
ki_candidates = ambiguous.get('ki', [])
print(f"'ki' has {len(ki_candidates)} candidate(s):")
for e in ki_candidates:
    print(f"  {e.eid}  {e.get_first('.GLOSS'):20s}  POS: {e.get_first('.POS')}")

print()
for prev in ['VB-STM', 'N', 'P-CASE-GEN']:
    winner = disambiguate(ki_candidates, prev)
    print(f"  prev_tag={prev:12s} → winner: {winner.eid}  ({winner.get_first('.POS')})")

---
## 3. Pass 1 — Standard Annotation

For each unannotated corpus line, the script extracts the word form, looks it up in
`form_to_lemma` / `form_to_candidates`, and inserts the chosen ID between the syntactic
path and the writing-mode tag.

In [ ]:
from oncoj.core.corpus import CorpusDocument

doc = CorpusDocument.from_file(CORPUS_FILE)

hits_existing = []   # (original_line, annotated_line, lemma_id)
hits_miss     = []   # word forms not found

for cl in doc.all_corpus_lines():
    if cl.is_annotated:
        continue
    form = cl.word_form
    prev_tag = cl.fields[-2] if len(cl.fields) >= 2 else ''

    if form in unambiguous:
        entry = unambiguous[form]
    elif form in ambiguous:
        entry = disambiguate(ambiguous[form], prev_tag)
    else:
        hits_miss.append(form)
        continue

    hits_existing.append((str(cl), entry.eid))

print(f"Pass 1 results for {CORPUS_FILE}:")
print(f"  Dictionary hits (would annotate): {len(hits_existing)}")
print(f"  Not in dictionary (pass 2):       {len(set(hits_miss))} distinct forms")

print("\nSample annotations (first 8):")
for orig, lid in hits_existing[:8]:
    print(f"  {lid}  ←  {orig[:80]}")

---
## 4. Pass 2 — Unknown Words (New IDs)

After Pass 1, any remaining unannotated tokens get a brand-new ID allocated by
`IDGenerator`, prefixed with the configured `LEMMA_PREFIX` (default `N`).

A minimal dictionary entry is also created for each new word.

In [ ]:
from oncoj.core.lemma_id import IDGenerator, LemmaID
from oncoj.core.kana import phonemic_to_kana

# Generator starts after the highest existing ID
all_ids = {LemmaID.parse(e.eid) for e in d.values()}
gen = IDGenerator(existing=all_ids, prefix='N')

unknown_forms = sorted(set(hits_miss))  # from pass 1 above
new_entries_preview = []
for form in unknown_forms[:5]:
    new_id = gen.next_id()
    kana   = phonemic_to_kana(form)
    new_entries_preview.append((str(new_id), form, kana))

print(f"Forms that would get new IDs (showing first 5 of {len(unknown_forms)}):")
print(f"{'New ID':12s}  {'Form':15s}  Kana")
print('-' * 45)
for nid, form, kana in new_entries_preview:
    print(f"{nid:12s}  {form:15s}  {kana}")

---
## 5. Dictionary Normalisation

The script also ensures every entry has its fields in the canonical order:
`.GLOSS → .MEANING → .FORM → .KANA → .POS` (then optional fields).

Use `Dictionary.from_file()` then check for non-canonical ordering:

In [ ]:
from oncoj.core.tags import REQUIRED_FIELDS

out_of_order = []
for entry in d.values():
    fields_present = [f for f in REQUIRED_FIELDS if entry.get_first(f) is not None]
    actual_order   = [k for k in entry._fields if k in set(REQUIRED_FIELDS)]
    if actual_order != fields_present:
        out_of_order.append(entry.eid)

if out_of_order:
    print(f"{len(out_of_order)} entries have non-canonical field order (sample):")
    for eid in out_of_order[:5]:
        print(f"  {eid}")
else:
    print("All entries already have canonical field order.")

# Show canonical order
print("\nCanonical required field order:")
for i, f in enumerate(REQUIRED_FIELDS, 1):
    print(f"  {i}. {f}")

---
## 6. Report Structure

After a real run, four report files are written to `OUTPUT_FOLDER`:

| File | Content |
|---|---|
| `report1_new_id_lines.txt` | Lines modified by a newly generated ID (Pass 2) |
| `report1_5_existing_id_lines.txt` | Lines modified by an existing dictionary ID (Pass 1) |
| `report2_normalized_entries.txt` | Dictionary entries reordered to canonical form |
| `report3_new_entries.txt` | New dictionary entries created |

Below is a simulated summary using the EN_01.txt data:

In [ ]:
all_files = sorted(os.listdir(TEXT_DIR))
total_lines   = 0
total_hits    = 0
total_unknown = 0

for fname in all_files:
    if not fname.endswith('.xml'):
        continue
    doc_i = CorpusDocument.from_file(os.path.join(TEXT_DIR, fname))
    lines = doc_i.all_corpus_lines()
    total_lines += len(lines)
    for cl in lines:
        if cl.is_annotated:
            continue
        form = cl.word_form
        if form in unambiguous or form in ambiguous:
            total_hits += 1
        else:
            total_unknown += 1

print("Corpus-wide annotation summary (dry run, no files written):")
print(f"  Total corpus lines  : {total_lines:6d}")
print(f"  Would get dict ID   : {total_hits:6d}  (Report 1.5)")
print(f"  Would get new ID    : {total_unknown:6d}  (Report 1)")


---
## 7. Running the Script

To actually process files, configure the `USER SETTINGS` block at the top of
`scripts/lemmas_processor.py`, then run:

```bash
python3 scripts/lemmas_processor.py
```

Key settings:

| Setting | Default | Effect |
|---|---|---|
| `OVERWRITE_SOURCE` | `False` | Write `*_processed.txt` to `OUTPUT_FOLDER` instead of editing source |
| `DICT_ENTRY_CREATE` | `True` | Create a new entry for each unknown word |
| `ADVANCED_DISAMBIG` | `True` | Use preceding-tag heuristic to resolve ambiguous forms |
| `NORMALIZE_DICT` | `True` | Rewrite dictionary to canonical field order |
| `AUTO_POS_QUERY` | `"N"` | Only annotate lines whose POS tag column is `N` (noun) |
| `LEMMA_PREFIX` | `"N"` | Prefix for newly generated IDs |